In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn import ensemble
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn import svm
from sklearn.svm import SVC

In [2]:
# SMSSpamCollection file уншиж авах хэсэг
df = pd.read_table('SMSSpamCollection', 
                   sep='\t', 
                   header=None, 
                   names=['label', 'sms_message'])

print('Мөрийн тоо:', df.shape[0])
print('Баганын тоо:', df.shape)
df.head()

Мөрийн тоо: 5572
Баганын тоо: (5572, 2)


,label,sms_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# **Data Preprocessing**

In [3]:
# ham болон span -р эхэлсэн хэсгүүдийг 0, 1 болгон хөрвүүлнэ.
df['label'] = df.label.map({'ham':0, 'spam':1})
df.head()

,label,sms_message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# харгалзах утгуудыг оноож өгнө. Жишээ нь: X_train хэсэгт sms_message баганыг г.м
X_train, X_test, y_train, y_test = train_test_split(df['sms_message'], 
                                                    df['label'], 
                                                    random_state=1,
                                                   test_size=0.3)

print('Мөр бичлэгийн тоо: {}'.format(df.shape[0]))
print('Сургалтын дата доторх өгөгдлийн тоо: {}'.format(X_train.shape[0]))
print('Өгөгдлийг турших өгөгдлийн цэгийн тоо : {}'.format(X_test.shape[0]))

Мөр бичлэгийн тоо: 5572
Сургалтын дата доторх өгөгдлийн тоо: 3900
Өгөгдлийг турших өгөгдлийн цэгийн тоо : 1672


# Үгсийн багцыг мэдээллийн санд ашиглах
Одоо бид өгөгдлөө хуваасан тул бидний дараагийн зорилго бол үгсийг хүссэн матрицын формат руу хөрвүүлэх явдал юм. Үүнийг хийхийн тулд бид CountVectorizer () -ийг ашиглах болно. Энд хоёр алхамыг анхаарч үзэх хэрэгтэй.


*   Нэгдүгээрт, бид сургалтын өгөгдлөө (X_train) CountVectorizer () -т багтааж, матрицыг буцааж өгөх ёстой.

*   Хоёрдугаарт, бид матрицыг буцаахын тулд тестийн өгөгдлөө (X_test) өөрчлөх хэрэгтэй.

X_train бол манай мэдээллийн бааз дахь 'sms_message' баганад зориулсан сургалтын өгөгдөл бөгөөд үүнийг ашиглан загвараа сургах болно.
X_test бол 'sms_message' баганын туршилтын өгөгдөл бөгөөд энэ нь урьдчилан таамаглахад ашиглагдах өгөгдөл юм (матрицад шилжсэний дараа). Дараа нь бид эдгээр таамаглалыг дараагийн шатанд y_test-тэй харьцуулах болно.

In [5]:
# Tf-idf аргыг ашиглах
count_vector = TfidfVectorizer()

# Сургалтын өгөгдлийг тохируулаад дараа нь матрицыг буцаана (Текстийн өгөгдлийн багц (мессеж) -ийг үгийн давтамжийн матриц болгон хөрвүүлэх)
training_data = (count_vector.fit_transform(X_train)).toarray()
print(training_data)
print(training_data.shape)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(3900, 7155)


**Туршилтын өгөгдлийг хувиргаж, матрицыг буцаана.**

*   Бид тестийн өгөгдлийг TfidfVectorizer () -д тохируулахгүй байгааг анхаарна уу.



In [6]:
testing_data = (count_vector.transform(X_test)).toarray()
print(testing_data)
print(testing_data.shape)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(1672, 7155)


**Сургалтын модел**

In [7]:
# Бидний загварыг бэлэн болгох
naive_bayes = MultinomialNB()
# Сургалтын өгөгдөлд манай загварыг тохируулах
naive_bayes.fit(training_data, y_train)

MultinomialNB()

**Урьдчилан таамаглах(Predict)**

In [8]:
# Урьдчилан таамаглах тест өгөгдөл
predictions = naive_bayes.predict(testing_data)

In [9]:
print('Гүйцэтгэл(Accuracy) оноо: ', format(accuracy_score(y_test, predictions)))
print('Нарийвчлал(Precision) оноо: ', format(precision_score(y_test, predictions)))
print('Recall оноо: ', format(recall_score(y_test, predictions)))
print('F1 дүн: ', format(f1_score(y_test, predictions)))

Гүйцэтгэл(Accuracy) оноо:  0.9575358851674641
Нарийвчлал(Precision) оноо:  1.0
Recall оноо:  0.691304347826087
F1 дүн:  0.8174807197943444


In [11]:
#  200 сул суралцагч (n_estimators) болон бусад бүх зүйлийг анхдагч утга болгон ашиглах

Bag = BaggingClassifier(n_estimators=200, n_jobs=-1)

# RandomForestClassifier програмыг ашиглан:
# 200 сул суралцагч (n_estimators) болон бусад бүх зүйлийг анхдагч утга болгон ашиглах

RF = RandomForestClassifier(n_estimators=200, n_jobs=-1)

# AdaBoostClassifier-ийг дараах байдлаар ашиглана:
# 300 сул суралцагчтай (n_estimators) ба суралцах түвшин 0.2 байна

ADBoost = AdaBoostClassifier(n_estimators=300, learning_rate=0.2)

# Анхдагч параметрийн утгатай шугаман SVM-ийг тохируулах:
SVM = SVC()

In [12]:
# BaggingClassifier-ээ сургалтын өгөгдөлд тохируулна
Bag.fit(training_data, y_train)

BaggingClassifier(n_estimators=200, n_jobs=-1)

In [13]:
# AdaBoostClassifier програмаа сургалтын өгөгдөлд тохируулна
ADBoost.fit(training_data, y_train)

/home/razydave/anaconda3/envs/pt_env/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoostClassifier(learning_rate=0.2, n_estimators=300)

In [14]:
# SVM-ээ сургалтын өгөгдөлд нийцүүлэх
SVM.fit(training_data, y_train)

SVC()

In [15]:
# RandomForestClassifier програмаа сургалтын өгөгдөлд тохируулна
RF.fit(training_data, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1)

In [16]:
# Туршилтын өгөгдөл дээр BaggingClassifier ашиглан урьдчилан таамаглах
y_Bag = Bag.predict(testing_data)

# Туршилтын өгөгдөлд RandomForestClassifier ашиглан урьдчилан таамаглах
y_RF = RF.predict(testing_data)

# Туршилтын өгөгдөлд AdaBoostClassifier ашиглан урьдчилан таамаглах
y_ADBoost = ADBoost.predict(testing_data)

# Туршилтын өгөгдөлд SVM ашиглан урьдчилан таамаглах
y_SVM = SVM.predict(testing_data)

In [17]:
def print_metrics(y_true, preds, model_name=None):
    '''
    Оролт:
    y_true = y_test = Өгөгдлийн баазад үнэн байх y утга (numpy array or pandas series)
    preds = predicted value (y_Bag, y_RF, y_ADBoost) = the predictions for those values from some model (numpy array or pandas series)
    model_name = BaggingClassifier/RandomForestClassifier/AdaBoostClassifier = a name associated with the model if you would like to add it to the print statements 
    
    Гаралт:
    accuracy(гүйцэтгэл), precision(нарийвчлал), recall, болон F1 оноог хэвлэнэ
    '''
    if model_name == None:
        print('Accuracy оноо: ', format(accuracy_score(y_true, preds)))
        print('Precision оноо: ', format(precision_score(y_true, preds)))
        print('Recall оноо: ', format(recall_score(y_true, preds)))
        print('F1 оноо: ', format(f1_score(y_true, preds)))
        print('\n\n')
    
    else:
        print('Accuracy оноо - ' + model_name + ' :' , format(accuracy_score(y_true, preds)))
        print('Precision оноо - ' + model_name + ' :', format(precision_score(y_true, preds)))
        print('Recall оноо - ' + model_name + ' :', format(recall_score(y_true, preds)))
        print('F1 оноо - ' + model_name + ' :', format(f1_score(y_true, preds)))
        print('\n\n')

In [18]:
# Print Bagging scores
print_metrics(y_true=y_test, preds=y_Bag, model_name="Bagging машин сургалтын арга")

# Print Random Forest scores
print_metrics(y_true=y_test, preds=y_RF, model_name="Randomforest машин сургалтын арга")

# Print AdaBoost scores
print_metrics(y_true=y_test, preds=y_ADBoost, model_name="AdaBoost машин сургалтын арга")

# Naive Bayes Classifier scores
print_metrics(y_true=y_test, preds=predictions, model_name="Naive Bayes машин сургалтын арга")

# SVM Classifier scores

print('Accuracy score for SVM :' , format(accuracy_score(y_test, y_SVM)))
print('Precision score for SVM :', format(precision_score(y_test, y_SVM, average= 'weighted', labels=np.unique(y_SVM))))
print('Recall score for SVM :', format(recall_score(y_test, y_SVM, average= 'weighted', labels=np.unique(y_SVM))))
print('F1 score for SVM :', format(f1_score(y_test, y_SVM, average= 'weighted', labels=np.unique(y_SVM))))

Accuracy оноо - Bagging машин сургалтын арга : 0.9694976076555024
Precision оноо - Bagging машин сургалтын арга : 0.9323671497584541
Recall оноо - Bagging машин сургалтын арга : 0.8391304347826087
F1 оноо - Bagging машин сургалтын арга : 0.8832951945080092



Accuracy оноо - Randomforest машин сургалтын арга : 0.9766746411483254
Precision оноо - Randomforest машин сургалтын арга : 0.9897435897435898
Recall оноо - Randomforest машин сургалтын арга : 0.8391304347826087
F1 оноо - Randomforest машин сургалтын арга : 0.908235294117647



Accuracy оноо - AdaBoost машин сургалтын арга : 0.9760765550239234
Precision оноо - AdaBoost машин сургалтын арга : 0.9523809523809523
Recall оноо - AdaBoost машин сургалтын арга : 0.8695652173913043
F1 оноо - AdaBoost машин сургалтын арга : 0.9090909090909091



Accuracy оноо - Naive Bayes машин сургалтын арга : 0.9575358851674641
Precision оноо - Naive Bayes машин сургалтын арга : 1.0
Recall оноо - Naive Bayes машин сургалтын арга : 0.691304347826087
F1 о